In [2]:
from pathlib import Path
import csv
import sys
import numpy as np

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.preprocessing import (
    MISSING_CATEGORY,
    UNKNOWN_CATEGORY,
    CALCULATED_FEATURES,
    NUMERICAL_FEATURES,
    ORDINAL_FEATURES,
    CATEGORICAL_FEATURES,
    encode_structural_missing,
    replace_missing_codes,
    numerical_medians,
    fit_ordinal_imputer,
    apply_imputer,
    standardization_stats,
    apply_standardization,
    fit_categorical_encoder,
    apply_categorical_encoder,
    add_intercept,
)

DATA_DIR = PROJECT_ROOT / "data" / "dataset"

In [3]:
with open(DATA_DIR / "x_train.csv", "r") as f:
    reader = csv.reader(f)
    all_headers = np.array(next(reader))

calc_indices = np.array([
    np.where(all_headers == feature)[0][0]
    for feature in CALCULATED_FEATURES
])

x_train_calc = np.genfromtxt(
    DATA_DIR / "x_train.csv",
    delimiter=",",
    skip_header=1,
    usecols=calc_indices,
    dtype=np.float32,
)

x_test_calc = np.genfromtxt(
    DATA_DIR / "x_test.csv",
    delimiter=",",
    skip_header=1,
    usecols=calc_indices,
    dtype=np.float32,
)

headers_calc = all_headers[calc_indices]

In [4]:
y_train = np.genfromtxt(
    DATA_DIR / "y_train.csv",
    delimiter=",",
    skip_header=1,
    usecols=1,
    dtype=int,
)

[autoreload of src.preprocessing failed: Traceback (most recent call last):
  File "/opt/jlab-env/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/jlab-env/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/opt/jlab-env/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1133, in get_code
  File "<frozen importlib._bootstrap_external>", line 1063, in source_to_code
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/home/ML_project_1/src/preprocessing.py", line 3
    x_train_final = 
                    ^
SyntaxError: invalid syntax
]

## Replacing missing values

In [5]:
x_train_structural = encode_structural_missing(
    x_train_calc,
    headers_calc,
)

x_test_structural = encode_structural_missing(
    x_test_calc,
    headers_calc,
)

x_train_clean = replace_missing_codes(
    x_train_structural,
    headers_calc,
)

x_test_clean = replace_missing_codes(
    x_test_structural,
    headers_calc,
)

In [6]:
def missingness_report(x, headers):
    report = []

    n_rows = x.shape[0]

    for idx, feature in enumerate(headers):
        n_missing = np.isnan(x[:, idx]).sum()
        pct_missing = 100 * n_missing / n_rows

        report.append(
            (feature, n_missing, pct_missing)
        )

    report.sort(key=lambda item: item[2], reverse=True)

    return report

In [7]:
missing_report = missingness_report(
    x_train_clean,
    headers_calc
)

for feature, n_missing, pct_missing in missing_report:
    print(
        f"{feature:12s} "
        f"{n_missing:8d} "
        f"{pct_missing:6.2f}%"
    )

PADUR2_        184643  56.27%
PAFREQ2_       182913  55.74%
PAMIN21_       119004  36.27%
_HCVU651       117708  35.87%
PAMIN11_       117355  35.76%
PAVIG21_       117012  35.66%
_MINAC21       116821  35.60%
_MINAC11       115152  35.09%
ACTIN21_       114698  34.95%
PADUR1_        113793  34.68%
PAVIG11_       113785  34.68%
METVL21_       112425  34.26%
PAFREQ1_       111257  33.91%
ACTIN11_       111072  33.85%
METVL11_       108727  33.13%
_INCOMG         58594  17.86%
_RFCHOL         46324  14.12%
_AIDTST3        42783  13.04%
_PACAT1         41344  12.60%
_PA300R2        41344  12.60%
_PA150R2        39414  12.01%
_VEGESUM        37605  11.46%
STRFREQ_        32496   9.90%
_FRUTSUM        32115   9.79%
_RFSEAT2        31643   9.64%
_RFSEAT3        31643   9.64%
_TOTINDA        28067   8.55%
_BMI5           27073   8.25%
WTKG3           23006   7.01%
_DRNKWEK        18967   5.78%
_RFDRHV5        18967   5.78%
_RFBING5        18858   5.75%
_PNEUMO2        17212   5.25%
DROCDY3_  

## Stratified data split

In [8]:
def stratified_train_val_split(x, y, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)

    train_indices = []
    val_indices = []

    for label in np.unique(y):
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)

        n_val = int(len(label_indices) * val_ratio)

        val_indices.extend(label_indices[:n_val])
        train_indices.extend(label_indices[n_val:])

    train_indices = np.array(train_indices)
    val_indices = np.array(val_indices)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)

    return (
        x[train_indices],
        x[val_indices],
        y[train_indices],
        y[val_indices],
    )

In [9]:
x_train_split, x_val_split, y_train_split, y_val_split = (
    stratified_train_val_split(
        x_train_clean,
        y_train,
        val_ratio=0.2,
        seed=42,
    )
)

In [10]:
print(x_train_split.shape)
print(x_val_split.shape)

print("Train positive rate:", np.mean(y_train_split == 1))
print("Validation positive rate:", np.mean(y_val_split == 1))

(262508, 49)
(65627, 49)
Train positive rate: 0.08830207079403295
Validation positive rate: 0.08830207079403295


## Numerical features imputation

In [11]:
numerical_medians = numerical_medians(
    x_train_split,
    headers_calc,
    NUMERICAL_FEATURES,
)

x_train_proc = apply_imputer(
    x_train_split,
    headers_calc,
    numerical_medians,
)

x_val_proc = apply_imputer(
    x_val_split,
    headers_calc,
    numerical_medians,
)

x_test_proc = apply_imputer(
    x_test_clean,
    headers_calc,
    numerical_medians,
)

## Ordinal features imputation

In [12]:
ordinal_modes = fit_ordinal_imputer(
    x_train_split,
    headers_calc,
    ORDINAL_FEATURES,
)
x_train_proc = apply_imputer(
    x_train_proc,
    headers_calc,
    ordinal_modes,
)
x_val_proc = apply_imputer(
    x_val_proc,
    headers_calc,
    ordinal_modes,
)
x_test_proc = apply_imputer(
    x_test_proc,
    headers_calc,
    ordinal_modes,
)

## Numerical and ordinal features standardization

In [13]:
FEATURES_TO_STANDARDIZE = (
    NUMERICAL_FEATURES
    + ORDINAL_FEATURES
)
standardization_stats = standardization_stats(
    x_train_proc,
    headers_calc,
    FEATURES_TO_STANDARDIZE,
)

x_train_proc = apply_standardization(
    x_train_proc,
    headers_calc,
     standardization_stats,
)

x_val_proc = apply_standardization(
    x_val_proc,
    headers_calc,
     standardization_stats,
)

x_test_proc = apply_standardization(
    x_test_proc,
    headers_calc,
     standardization_stats,
)

## Categorical Data One-Hot Encoding

In [14]:
categorical_encoder = fit_categorical_encoder(
    x_train_proc,
    headers_calc,
    CATEGORICAL_FEATURES,
)
x_train_encoded, headers_encoded = apply_categorical_encoder(
    x_train_proc,
    headers_calc,
    CATEGORICAL_FEATURES,
    categorical_encoder,
)

x_val_encoded, headers_val = apply_categorical_encoder(
    x_val_proc,
    headers_calc,
    CATEGORICAL_FEATURES,
    categorical_encoder,
)

x_test_encoded, headers_test = apply_categorical_encoder(
    x_test_proc,
    headers_calc,
    CATEGORICAL_FEATURES,
    categorical_encoder,
)

In [15]:
assert np.array_equal(headers_encoded, headers_val)
assert np.array_equal(headers_encoded, headers_test)

assert x_train_encoded.shape[1] == x_val_encoded.shape[1]
assert x_train_encoded.shape[1] == x_test_encoded.shape[1]

In [16]:
def find_unseen_categories(
    x,
    headers,
    categorical_features,
    encoder,
):
    unseen = {}

    for feature in categorical_features:
        idx = np.where(headers == feature)[0][0]

        column = x[:, idx].copy()
        column[np.isnan(column)] = MISSING_CATEGORY

        known = encoder[feature]["categories"]
        known = known[known != UNKNOWN_CATEGORY]

        unseen_values = np.unique(
            column[~np.isin(column, known)]
        )

        if len(unseen_values) > 0:
            unseen[feature] = unseen_values

    return unseen

In [17]:
print(
    "Validation:",
    find_unseen_categories(
        x_val_proc,
        headers_calc,
        CATEGORICAL_FEATURES,
        categorical_encoder,
    )
)

print(
    "Test:",
    find_unseen_categories(
        x_test_proc,
        headers_calc,
        CATEGORICAL_FEATURES,
        categorical_encoder,
    )
)

Validation: {}
Test: {}


In [18]:
x_train_final = add_intercept(x_train_encoded)
x_val_final = add_intercept(x_val_encoded)
x_test_final = add_intercept(x_test_encoded)

headers_final = np.concatenate([
    np.array(["intercept"]),
    headers_encoded
])

In [19]:
print("Train:", x_train_final.shape)
print("Validation:", x_val_final.shape)
print("Test:", x_test_final.shape)

assert x_train_final.shape[1] == x_val_final.shape[1]
assert x_train_final.shape[1] == x_test_final.shape[1]

assert not np.isnan(x_train_final).any()
assert not np.isnan(x_val_final).any()
assert not np.isnan(x_test_final).any()

assert not np.isinf(x_train_final).any()
assert not np.isinf(x_val_final).any()
assert not np.isinf(x_test_final).any()

Train: (262508, 101)
Validation: (65627, 101)
Test: (109379, 101)


In [20]:
print(x_train_final)

[[1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]]


In [21]:
print(x_val_final)
print(x_test_final)
    headers_final,
    y_train_split,
    y_val_split, 

IndentationError: unexpected indent (2130859654.py, line 2)